In [1]:

import numpy as np
import pandas as pd
import os
import rioxarray as rxr
from osgeo import gdal
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline, make_pipeline
import geopandas as gpd
import rasterio
from shapely.geometry import box
from shapely.geometry import Point
from rasterio.sample import sample_gen
import xgboost as xgb

from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from skopt import BayesSearchCV
from skopt.space import Integer, Real


In [10]:
def save_img(output_filename, reference_filename, array):
    
    reference = gdal.Open(reference_filename)
    
    # 获取地理变换参数和坐标系
    geo_transform = reference.GetGeoTransform()
    projection = reference.GetProjection()

    # 获取矩阵尺寸
    rows, cols = array.shape

    # 创建新的 TIFF 数据集
    driver = gdal.GetDriverByName('GTiff')
    out_data = driver.Create(output_filename, cols, rows, 1, gdal.GDT_Float32)

    # 设置地理变换和坐标系
    out_data.SetGeoTransform(geo_transform)
    out_data.SetProjection(projection)
    

    # 写入数据
    out_data.GetRasterBand(1).WriteArray(array)

    # 保存并关闭文件
    out_data.FlushCache()
    out_data = None

In [ ]:
path = 'G:\FB_new7\Region3b1_new7_\Region3a1_2022_FB6_new7.tif'


    

k = rxr.open_rasterio(path)
k.shape
Freq = k[0]
NDWI_min = k[4]
mask = (Freq>0.1) & (Freq<0.9) & (NDWI_min < -0.1)



(7, 4151, 1826)

In [18]:
import rioxarray as rxr
import numpy as np
import xarray as xr

# 读取影像
path = r'G:\FB_new7\Region3a1_new7_\Region3b1_2022_FB6_new7.tif'
path = r'G:\ICESat-2\Region4_map\Solway_map\A2019\Solway_2019_FB6_new2.tif'
k = rxr.open_rasterio(path, masked=True)

# 提取需要的波段
Freq = k[0]
NDWI_min = k[1]

# 构建掩模（注意掩模为True的区域保留）
mask = (Freq > 0.05) & (Freq < 0.95) & (NDWI_min < -0.1)
# mask = (Freq > 0.05) & (Freq < 0.95) & (NDWI_min < -0.1)
# 应用掩模到整个数据（保留符合条件区域，其余设为NaN）
k_masked = k.where(mask)

# 设置掩模结果的 nodata 值（可选）
k_masked.rio.write_nodata(0, inplace=True)

# 指定输出路径
output_path = r'G:\ICESat-2\Region4_map\Solway_map\A2019\Solway_2019_FB6_new2_masked.tif'

# 写出为 GeoTIFF
k_masked.rio.to_raster(output_path)
